# L12 — Project 4: Beat Tracker

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yanluo/stem-on-stage-notebooks/blob/main/L12/04_beat_tracker/beat_starter.ipynb)

**Goal:** estimate a dancer's tempo (beats per minute) from accelerometer data, so the lanterns can pulse *with* the dance instead of running on a fixed schedule.

**Two methods, side by side:** peak counting (intuitive) and autocorrelation (more robust). The whole notebook works on a bundled 30-second recording where the tempo steps from 120 BPM down to 100 BPM at the 20-second mark — a plot that *shows* the change is the point of the day.

> **New to pandas / numpy / scipy?** Skim [`L12/00_python_data_tools/python_data_tools_starter.ipynb`](../00_python_data_tools/python_data_tools_starter.ipynb) first — it's a 30-minute tour of every function this notebook uses, with tiny standalone examples.

## Step 0 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, correlate

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

## Step 1 — Load the recording

In [ ]:
SAMPLE_URL = "https://raw.githubusercontent.com/yanluo/stem-on-stage-notebooks/main/data/sample-beat.csv"

if IN_COLAB:
    df = pd.read_csv(SAMPLE_URL)
else:
    df = pd.read_csv("../../data/sample-beat.csv")

df["mag"] = np.sqrt(df["x"]**2 + df["y"]**2 + df["z"]**2)
HZ = 50            # sample rate of this recording
print(df.shape)
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(df["t"], df["mag"], color="purple", linewidth=0.7)
ax.axvline(20, color="gray", linestyle="--", label="tempo change")
ax.set_xlabel("time (s)")
ax.set_ylabel("|a| (mg)")
ax.legend()
plt.show()

# Method A — Peak counting

## Step 2 — Find every beat

If we can locate the magnitude peak of each beat, the average gap between peaks is the period — and `60 / period` is BPM. `distance=` rejects peaks closer than 0.3 s apart, which caps detectable BPM at 200 (plenty for dance).

In [ ]:
peak_idx, _ = find_peaks(df["mag"], height=df["mag"].mean(),
                          distance=int(0.3 * HZ))
peak_times = df["t"].iloc[peak_idx].to_numpy()
print(f"{len(peak_times)} peaks detected")

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(df["t"], df["mag"], color="purple", linewidth=0.5)
ax.scatter(peak_times, df["mag"].iloc[peak_idx], s=20, color="red", zorder=3, label="detected beats")
ax.set_xlabel("time (s)"); ax.set_ylabel("|a| (mg)"); ax.legend()
plt.show()

## Step 3 — Average BPM across the whole recording

In [ ]:
intervals = np.diff(peak_times)
mean_period = intervals.mean()
print(f"average gap between beats: {mean_period:.3f} s")
print(f"average BPM: {60 / mean_period:.1f}")

## Step 4 — BPM over time

The recording's tempo changes at t=20 s. A single average smears that out. Slide a 5-second window across the peaks, compute BPM in each window, and plot.

In [ ]:
WINDOW_S = 5.0
window_centers, window_bpms = [], []
for center in np.arange(WINDOW_S/2, df["t"].max() - WINDOW_S/2, 1.0):
    in_window = (peak_times >= center - WINDOW_S/2) & (peak_times <= center + WINDOW_S/2)
    window_peaks = peak_times[in_window]
    if len(window_peaks) >= 2:
        window_centers.append(center)
        window_bpms.append(60.0 / np.diff(window_peaks).mean())

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(window_centers, window_bpms, "o-", color="steelblue", label="inferred")
ax.axhline(120, color="green", linestyle="--", label="true 120 BPM")
ax.axhline(100, color="orange", linestyle="--", label="true 100 BPM")
ax.axvline(20, color="gray", linestyle="--")
ax.set_xlabel("time (s)"); ax.set_ylabel("BPM"); ax.legend()
plt.show()

# Method B — Autocorrelation

## Step 5 — A more robust way

Peak counting depends on the height threshold. **Autocorrelation** asks a different question: "how much does this signal look like itself, shifted by `lag` samples?" The lag where it most resembles itself *is* the period.

Subtract the mean first so the autocorrelation peaks are clean.

In [ ]:
# Just the first 20 s (the 120-BPM section) to keep it tidy.
mag = df[df["t"] < 20]["mag"].to_numpy()
mag = mag - mag.mean()
ac = correlate(mag, mag, mode="full")
ac = ac[len(ac)//2:]                       # keep non-negative lags only
ac = ac / ac[0]                            # normalize so lag-0 = 1
lags_s = np.arange(len(ac)) / HZ

# Skip lag 0 itself; cap search to a sensible BPM range.
search = (lags_s > 0.3) & (lags_s < 1.5)   # 40-200 BPM
best_lag_s = lags_s[search][np.argmax(ac[search])]
print(f"best autocorrelation lag: {best_lag_s:.3f} s")
print(f"BPM (autocorr): {60 / best_lag_s:.1f}")

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(lags_s, ac, color="crimson")
ax.axvline(best_lag_s, color="black", linestyle="--", label=f"period = {best_lag_s:.3f} s")
ax.set_xlim(0, 2)
ax.set_xlabel("lag (s)"); ax.set_ylabel("autocorrelation"); ax.legend()
plt.show()

## What to build next (L13 starting goals)

1. **Live wearable.** Flash [`wearable_beat.py`](wearable_beat.py): the device runs peak detection in real time, blinks the LED on each beat, and broadcasts `radio.send_value("beat", n)` to the lanterns.
2. **Lantern integration.** Wire the `"beat"` radio messages into the L7 controller stack so a NeoPixel pulse fires on every detected beat.
3. **Compare against ground truth.** Look up the song's published BPM (Spotify shows it) and report the error of your detector.
4. **Filter before detecting.** Bandpass-filter the magnitude to 1–4 Hz before `find_peaks` — kills high-frequency jitter that produces false peaks.
5. **Phase-aware lighting.** Track *which* beat in a 4-beat measure you're on (every 4th beat = downbeat → brighter color).
6. **Recovery from drop-out.** What does your detector do when the dancer freezes for 2 seconds? Add a "hold last good tempo for X seconds, then declare unknown" rule.

## Reflect (homework)

Write 3–4 sentences:
- Which extension will you build first?
- What's a song with a known BPM you could test against?
- Peak counting or autocorrelation — which would you trust more for live use, and why?